# 🎵 ACE-Step 1.5 편집 화면 — 한 번에 설치 · 실행

1. **Kaggle**: 오른쪽 Settings 에서 *Internet on*, *Accelerator GPU T4* → 위 메뉴 **Run All**
2. **Colab**: 런타임 → 런타임 유형 변경 → T4 GPU → **모두 실행**
3. 5~10분 뒤 출력에 `🌐 접속 주소` 가 나오면 열면 된다(Kaggle 은 공개 주소로만 접속 가능).

끝나면 세션을 꺼서 GPU 시간을 아낀다. 원곡을 편곡한 결과는 권리 확인 전에 공개하지 않는다.

In [ ]:
#@title 1) 설치 — 셀을 실행만 하면 된다 (Kaggle: Internet 켜기 + GPU T4 / Colab: 런타임 T4)
MODEL = "acestep-v15-turbo"  #@param ["acestep-v15-turbo", "acestep-v15-xl-turbo", "acestep-v15-xl-base"]
# T4(16GB)에서는 turbo 를 권장. 원곡 선율을 지키는 편곡용 xl-base 는 24GB 이상 GPU 에서만.

import os, subprocess, sys, shutil
ON_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{BASE}/ACE-Step-1.5"

def sh(c, quiet=False):
    print("▶", c if len(c) < 120 else c[:117] + "...", flush=True)
    r = subprocess.run(c, shell=True, text=True, capture_output=quiet)
    if r.returncode != 0:
        if quiet: print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit(f"❌ 실패: {c}")

print("🎵 ACE-Step 1.5 설치 —", "Kaggle" if ON_KAGGLE else "Colab")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["MPLBACKEND"] = "Agg"          # 노트북의 matplotlib_inline 이 넘어가면 하위 프로세스가 죽는다

if not os.path.exists(REPO):
    sh(f"git clone --depth 1 -q https://github.com/ace-step/ACE-Step-1.5.git {REPO}")
os.chdir(REPO)
sh("pip install -q uv", quiet=True)
# T4 는 bf16 이 없어 float16 을 고르는데 편곡 · 악기 추가에서 NaN(무음 · 잡음) → float32
sh("sed -i 's/self.dtype = torch.float16/self.dtype = torch.float32/' acestep/core/generation/handler/init_service_orchestrator.py")
sh("uv sync -q", quiet=True)
sh("uv pip install -q hf_transfer", quiet=True)
print("✅ 설치 끝 — 아래 2) 셀을 실행하세요")


In [ ]:
#@title 2) 실행 — 모델을 올리고 공개 주소(https://…)를 띄운다
import os, re, subprocess, threading, time, shutil
os.chdir(REPO)
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    raise SystemExit("저장(Save Version) 실행에서는 화면을 띄우지 않는다 — 노트북을 열고 직접 실행하세요")

cmd = (f"uv run acestep --server-name 0.0.0.0 --port 7860 --share --init_service true --config_path {MODEL} "
       "--init_llm false --quantization none --offload_to_cpu true --offload_dit_to_cpu true")
print("▶", cmd, flush=True)
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = {"v": None}

def banner(u, how):
    print("\n" + "=" * 70 + f"\n🌐 접속 주소 ({how}):  {u}\n" + "=" * 70 + "\n", flush=True)

def cloudflared():
    """gradio.live 가 안 열리면 cloudflared 터널로 대신 연다."""
    if not shutil.which("cloudflared"):
        subprocess.run("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                       "&& chmod +x /usr/local/bin/cloudflared", shell=True)
    cf = subprocess.Popen("cloudflared tunnel --url http://127.0.0.1:7860 --no-autoupdate", shell=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in cf.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url["v"] = m.group(0); banner(url["v"], "cloudflared"); break

started = time.time(); local_up = False
for line in proc.stdout:
    print(line, end="", flush=True)
    if "Running on local URL" in line:
        local_up = True; local_t = time.time()
    m = re.search(r"https://[a-z0-9]+\.gradio\.live", line)
    if m and not url["v"]:
        url["v"] = m.group(0); banner(url["v"], "gradio share, 72시간")
    if local_up and not url["v"] and time.time() - local_t > 60:
        threading.Thread(target=cloudflared, daemon=True).start(); local_up = False
print("서버가 멈췄습니다. 위 로그의 오류를 확인하세요.")
